# H_DWF eigenmode TCD comparison

Compares six fermion TCD estimators — `q_{A,B,C}` with both `mgap` (w = m_gap/μ_n)
and `sign` (w = sign(μ_n)) weighting — against:

1. **Gluonic TCD** at Wilson-flow times TD_τ = 0, 4, 16  
2. **Luchang (qlat) stochastic DWF TCD** reference fields (index 0, 1, ...)

Data files are written by `Generate_data_mvi.sh §2.5` after running `FieldDensityEigen`.

## File formats
| file | columns |
|------|---------|
| `data/corr_ip_q_{X}_{w}.dat` | TD_tau  tau  conf  value  (Corr and IP interleaved) |
| `data/comp_ref_q_{X}_{w}.dat` | comp_idx  tau  conf  Q_evec  Q_ref  Corr  IP  rms_diff |
| `data/corr_ip_stoch.dat` | comp_idx  TD_tau  tau  conf  Q_ref  Q_gluon  Corr  IP |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# ---------- path to HMC output directory ----------
HMC_DIR = os.path.expanduser(
    '~/tmp/src/Grid_cleanedup_for_pullrequest/systems/Frontier/HMC/32cube-rho0.124-tau4'
)
DATA = os.path.join(HMC_DIR, 'data')

TD_TAUS   = [0, 4, 16]          # gluonic Wilson-flow times
WF_TAUS   = [0, 4]              # DWF Wilson-flow times
DEFS      = ['A', 'B', 'C']    # formula labels
WEIGHTS   = ['mgap', 'sign']   # weight modes
ALL_DEFS  = [f'{d}_{w}' for w in WEIGHTS for d in DEFS]  # A_mgap … C_sign
print('Definitions:', ALL_DEFS)

## 1. Load fermion TCD vs gluonic TCD (Pearson correlation)

In [ ]:
# corr_ip_q_{X}_{w}.dat  columns: TD_tau  tau  conf  value
# Rows alternate Corr / IP (same TD_tau, tau, conf pair repeated)

rows_gluon = []
for w in WEIGHTS:
    for d in DEFS:
        key = f'{d}_{w}'
        fpath = os.path.join(DATA, f'corr_ip_q_{key}.dat')
        if not os.path.exists(fpath):
            print(f'  missing: {fpath}')
            continue
        raw = np.loadtxt(fpath)
        if raw.ndim == 1:
            raw = raw.reshape(1, -1)
        # rows come in Corr/IP pairs: even rows = Corr, odd rows = IP
        corr_rows = raw[0::2]
        ip_rows   = raw[1::2]
        n = min(len(corr_rows), len(ip_rows))
        for i in range(n):
            td, tau, conf, corr_val = corr_rows[i]
            _,  _,   _,    ip_val  = ip_rows[i]
            rows_gluon.append(dict(
                formula=f'q_{d}', weight=w, def_key=f'q_{key}',
                TD_tau=int(td), tau=int(tau), conf=int(conf),
                Corr=corr_val, IP=ip_val
            ))

df_gluon = pd.DataFrame(rows_gluon)
print(f'Loaded {len(df_gluon)} rows (gluonic comparison)')
df_gluon.head()

## 2. Load fermion TCD vs Luchang (qlat) stochastic reference

In [ ]:
# comp_ref_q_{X}_{w}.dat  columns: comp_idx  tau  conf  Q_evec  Q_ref  Corr  IP  rms_diff

rows_qlat = []
for w in WEIGHTS:
    for d in DEFS:
        key = f'{d}_{w}'
        fpath = os.path.join(DATA, f'comp_ref_q_{key}.dat')
        if not os.path.exists(fpath):
            print(f'  missing: {fpath}')
            continue
        raw = np.loadtxt(fpath)
        if raw.ndim == 1:
            raw = raw.reshape(1, -1)
        for row in raw:
            ci, tau, conf, Q_evec, Q_ref, corr, ip, rms = row
            rows_qlat.append(dict(
                formula=f'q_{d}', weight=w, def_key=f'q_{key}',
                comp_idx=int(ci), tau=int(tau), conf=int(conf),
                Q_evec=Q_evec, Q_ref=Q_ref, Corr=corr, IP=ip, rms_diff=rms
            ))

df_qlat = pd.DataFrame(rows_qlat)
print(f'Loaded {len(df_qlat)} rows (qlat comparison)')
df_qlat.head()

## 3. Load qlat stochastic vs gluonic TCD

In [ ]:
# corr_ip_stoch.dat  columns: comp_idx  TD_tau  tau  conf  Q_ref  Q_gluon  Corr  IP
fpath_stoch = os.path.join(DATA, 'corr_ip_stoch.dat')
df_stoch = pd.DataFrame()
if os.path.exists(fpath_stoch):
    raw = np.loadtxt(fpath_stoch)
    if raw.ndim == 1: raw = raw.reshape(1, -1)
    df_stoch = pd.DataFrame(raw, columns=['comp_idx','TD_tau','tau','conf',
                                           'Q_ref','Q_gluon','Corr','IP'])
    df_stoch = df_stoch.astype({'comp_idx':int,'TD_tau':int,'tau':int,'conf':int})
    print(f'Loaded {len(df_stoch)} rows (qlat vs gluonic)')
    display(df_stoch.head())
else:
    print('corr_ip_stoch.dat not found')

## 4. Summary table: Pearson correlation, averaged over configs

Rows = formula × weight mode.  
Columns = comparison target (gluonic TD_τ=0/4/16, qlat stoch idx=0/1).  
Values = mean Pearson correlation across available configs.

In [ ]:
def make_summary_table(tau_filter=None):
    """
    Build a summary DataFrame:
      rows   = (formula, weight) pairs  e.g. (q_A, mgap)
      cols   = gluon_TD0 gluon_TD4 gluon_TD16  qlat_0 qlat_1  [qlat_stoch_TD0 qlat_stoch_TD4 ...]
      values = mean Pearson Corr  (across configs, tau_filter applied when given)
    """
    row_index = pd.MultiIndex.from_tuples(
        [(f'q_{d}', w) for w in WEIGHTS for d in DEFS],
        names=['formula', 'weight']
    )

    # ---- gluonic columns ----
    dg = df_gluon.copy()
    if tau_filter is not None:
        dg = dg[dg.tau == tau_filter]
    gluon_cols = {}
    for td in TD_TAUS:
        sub = dg[dg.TD_tau == td].groupby(['formula','weight'])['Corr'].mean()
        gluon_cols[f'gluon_TD{td}'] = sub

    # ---- qlat columns ----
    dq = df_qlat.copy()
    if tau_filter is not None:
        dq = dq[dq.tau == tau_filter]
    qlat_cols = {}
    for ci in sorted(dq.comp_idx.unique()) if len(dq) > 0 else []:
        sub = dq[dq.comp_idx == ci].groupby(['formula','weight'])['Corr'].mean()
        qlat_cols[f'qlat_{ci}'] = sub

    # ---- qlat-vs-gluon reference row (single row, not per fermion def) ----
    stoch_vals = {}
    if len(df_stoch) > 0:
        ds = df_stoch.copy()
        if tau_filter is not None:
            ds = ds[ds.tau == tau_filter]
        for td in TD_TAUS:
            for ci in sorted(ds.comp_idx.unique()):
                key = f'qlat{ci}_TD{td}'
                val = ds[(ds.TD_tau == td) & (ds.comp_idx == ci)]['Corr'].mean()
                stoch_vals[key] = val

    # ---- assemble table ----
    all_cols = list(gluon_cols.keys()) + list(qlat_cols.keys())
    df_tab = pd.DataFrame(index=row_index, columns=all_cols, dtype=float)
    for col, series in {**gluon_cols, **qlat_cols}.items():
        for (form, wt), val in series.items():
            if (form, wt) in df_tab.index:
                df_tab.loc[(form, wt), col] = val

    return df_tab, stoch_vals


# ---- All DWF-tau values combined ----
tbl_all, stoch_all = make_summary_table(tau_filter=None)
print('=== Pearson correlation (all tau, mean over configs) ===')
display(tbl_all.style
    .format('{:.3f}')
    .background_gradient(cmap='RdYlGn', vmin=-0.2, vmax=1.0, axis=None)
    .set_caption('Corr: fermion TCD definitions vs gluonic/qlat reference')
)

if stoch_all:
    print('\nqlat stochastic vs gluonic TCD (reference, no fermion def):')
    display(pd.Series(stoch_all).rename('Corr').to_frame().T.style.format('{:.3f}'))

In [ ]:
# ---- Per DWF-tau ----
for tau in WF_TAUS:
    tbl, stoch = make_summary_table(tau_filter=tau)
    print(f'\n=== Pearson correlation — DWF Wilson-flow tau={tau} ===')
    display(tbl.style
        .format('{:.3f}')
        .background_gradient(cmap='RdYlGn', vmin=-0.2, vmax=1.0, axis=None)
        .set_caption(f'tau={tau}')
    )
    if stoch:
        print(f'  qlat vs gluonic (tau={tau}):', {k: f'{v:.3f}' for k, v in stoch.items()})

## 5. Summary table: normalised inner product

In [ ]:
def make_ip_table(tau_filter=None):
    row_index = pd.MultiIndex.from_tuples(
        [(f'q_{d}', w) for w in WEIGHTS for d in DEFS],
        names=['formula', 'weight']
    )
    dg = df_gluon.copy()
    if tau_filter is not None: dg = dg[dg.tau == tau_filter]
    dq = df_qlat.copy()
    if tau_filter is not None: dq = dq[dq.tau == tau_filter]

    gluon_cols = {f'gluon_TD{td}': dg[dg.TD_tau==td].groupby(['formula','weight'])['IP'].mean()
                  for td in TD_TAUS}
    qlat_cols  = {f'qlat_{ci}':    dq[dq.comp_idx==ci].groupby(['formula','weight'])['IP'].mean()
                  for ci in (sorted(dq.comp_idx.unique()) if len(dq) > 0 else [])}

    all_cols = list(gluon_cols.keys()) + list(qlat_cols.keys())
    df_tab = pd.DataFrame(index=row_index, columns=all_cols, dtype=float)
    for col, series in {**gluon_cols, **qlat_cols}.items():
        for (form, wt), val in series.items():
            if (form, wt) in df_tab.index:
                df_tab.loc[(form, wt), col] = val
    return df_tab

tbl_ip = make_ip_table()
print('=== Normalised inner product (all tau) ===')
display(tbl_ip.style
    .format('{:.3f}')
    .background_gradient(cmap='RdYlGn', vmin=-0.2, vmax=1.0, axis=None)
    .set_caption('IP: fermion TCD definitions vs gluonic/qlat reference')
)

## 6. Bar chart: Corr vs gluonic TCD — mgap vs sign, all formulas

In [ ]:
if len(df_gluon) > 0:
    fig, axes = plt.subplots(1, len(TD_TAUS), figsize=(5*len(TD_TAUS), 4), sharey=True)
    if len(TD_TAUS) == 1: axes = [axes]

    x = np.arange(len(DEFS))
    width = 0.35
    colors = {'mgap': '#1f77b4', 'sign': '#ff7f0e'}

    for ax, td in zip(axes, TD_TAUS):
        sub = df_gluon[df_gluon.TD_tau == td]
        for i, wt in enumerate(WEIGHTS):
            vals = [
                sub[(sub.formula==f'q_{d}') & (sub.weight==wt)]['Corr'].mean()
                for d in DEFS
            ]
            offset = (i - 0.5) * width
            ax.bar(x + offset, vals, width, label=wt, color=colors[wt], alpha=0.85)
        ax.set_title(f'Gluonic TCD TD_τ={td}')
        ax.set_xticks(x)
        ax.set_xticklabels([f'q_{d}' for d in DEFS])
        ax.set_ylabel('Pearson Corr')
        ax.axhline(0, color='k', linewidth=0.5)
        ax.legend()

    fig.suptitle('Fermion TCD estimators vs gluonic TCD  (mean over configs)', y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(DATA, 'corr_vs_gluon.pdf'), bbox_inches='tight')
    plt.show()

## 7. Bar chart: Corr vs qlat stochastic reference — mgap vs sign

In [ ]:
if len(df_qlat) > 0:
    n_qlat = df_qlat.comp_idx.nunique()
    fig, axes = plt.subplots(1, n_qlat, figsize=(5*n_qlat, 4), sharey=True)
    if n_qlat == 1: axes = [axes]

    x = np.arange(len(DEFS))
    width = 0.35
    colors = {'mgap': '#1f77b4', 'sign': '#ff7f0e'}

    for ax, ci in zip(axes, sorted(df_qlat.comp_idx.unique())):
        sub = df_qlat[df_qlat.comp_idx == ci]
        for i, wt in enumerate(WEIGHTS):
            vals = [
                sub[(sub.formula==f'q_{d}') & (sub.weight==wt)]['Corr'].mean()
                for d in DEFS
            ]
            offset = (i - 0.5) * width
            ax.bar(x + offset, vals, width, label=wt, color=colors[wt], alpha=0.85)
        ax.set_title(f'qlat stochastic ref idx={ci}')
        ax.set_xticks(x)
        ax.set_xticklabels([f'q_{d}' for d in DEFS])
        ax.set_ylabel('Pearson Corr')
        ax.axhline(0, color='k', linewidth=0.5)
        ax.legend()

    fig.suptitle('Fermion TCD estimators vs qlat stochastic TCD  (mean over configs)', y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(DATA, 'corr_vs_qlat.pdf'), bbox_inches='tight')
    plt.show()

## 8. Per-config scatter: Q_evec vs Q_ref (topological charge agreement)

In [ ]:
if len(df_qlat) > 0:
    # Show Q_evec vs Q_ref for q_B (best spatial estimator candidate)
    for wt in WEIGHTS:
        sub = df_qlat[(df_qlat.formula == 'q_B') & (df_qlat.weight == wt)]
        if len(sub) == 0: continue
        fig, ax = plt.subplots(figsize=(4, 4))
        sc = ax.scatter(sub.Q_ref, sub.Q_evec, c=sub.conf, cmap='viridis', alpha=0.8)
        lim = max(abs(sub.Q_ref).max(), abs(sub.Q_evec).max()) * 1.1
        ax.plot([-lim, lim], [-lim, lim], 'k--', alpha=0.5, label='y=x')
        ax.set_xlabel('Q_ref (qlat)')
        ax.set_ylabel('Q_evec (q_B)')
        ax.set_title(f'Topological charge: q_B_{wt} vs qlat')
        plt.colorbar(sc, ax=ax, label='conf')
        ax.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(DATA, f'Q_scatter_q_B_{wt}.pdf'), bbox_inches='tight')
        plt.show()

## 9. Full comparison table: Corr + IP, all defs, all references

In [ ]:
# Wide table: each metric (Corr, IP) × each reference column, all 6 defs
if len(df_gluon) > 0 or len(df_qlat) > 0:
    pieces = []

    # Gluonic Corr
    for td in TD_TAUS:
        sub = df_gluon[df_gluon.TD_tau == td].groupby(['formula','weight'])[['Corr','IP']].mean()
        sub.columns = pd.MultiIndex.from_tuples([(f'gluon_TD{td}','Corr'),(f'gluon_TD{td}','IP')])
        pieces.append(sub)

    # qlat Corr
    for ci in (sorted(df_qlat.comp_idx.unique()) if len(df_qlat) > 0 else []):
        sub = df_qlat[df_qlat.comp_idx==ci].groupby(['formula','weight'])[['Corr','IP']].mean()
        sub.columns = pd.MultiIndex.from_tuples([(f'qlat_{ci}','Corr'),(f'qlat_{ci}','IP')])
        pieces.append(sub)

    if pieces:
        df_full = pd.concat(pieces, axis=1)
        # Reorder: mgap rows first, sign rows second
        idx_order = [(f'q_{d}', w) for w in WEIGHTS for d in DEFS]
        df_full = df_full.reindex([i for i in idx_order if i in df_full.index])
        print('=== Full comparison table (mean over configs) ===')
        display(df_full.style
            .format('{:.3f}')
            .background_gradient(cmap='RdYlGn', vmin=-0.2, vmax=1.0,
                                  subset=pd.IndexSlice[:, pd.IndexSlice[:,'Corr']])
            .set_caption('Fermion TCD: Corr + IP vs gluonic and qlat references')
        )